# Notebook 09 — SOH Model Validation & Hyperparameter Tuning

**Objective:** Cross-validate SOH models using GroupKFold (by battery_id),
then tune the best models with RandomizedSearchCV.

**Why GroupKFold?**
Standard k-fold would allow training and validation cycles from the
**same battery** to appear in different folds. This inflates performance
estimates by leaking per-battery trends. GroupKFold ensures each fold
holds out ALL cycles from specific batteries.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, time

ROOT = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.join(ROOT, 'src'))

from evaluation import compute_metrics, build_comparison_table, print_metrics_table

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_validate

sns.set_style('whitegrid')

## 1. Prepare Training Data

In [ ]:
final = pd.read_csv(os.path.join(ROOT, 'cleaned_data', 'final_dataset.csv'))

rng = np.random.default_rng(42)
all_bats = sorted(final['battery_id'].unique())
n_test   = round(len(all_bats) * 0.30)
test_bats  = sorted(rng.choice(all_bats, size=n_test, replace=False).tolist())
train_bats = [b for b in all_bats if b not in test_bats]

soh_train = final[final['battery_id'].isin(train_bats)]
soh_test  = final[final['battery_id'].isin(test_bats)]

SOH_FEATURES = ['cycle', 'discharge_duration', 'energy_Wh',
                 'voltage_mean', 'voltage_min', 'voltage_std',
                 'current_mean', 'temperature_mean', 'temperature_max', 'ambient_temp_C']

X_tr = soh_train[SOH_FEATURES].values;  y_tr = soh_train['soh'].values
X_te = soh_test[SOH_FEATURES].values;   y_te = soh_test['soh'].values
groups = soh_train['battery_id'].values

print(f'Training rows: {len(X_tr)}  across {len(set(groups))} batteries')

## 2. GroupKFold Cross-Validation (5 Folds, by Battery)

In [ ]:
gkf = GroupKFold(n_splits=5)

models_cv = {
    'Linear Regression': Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'Random Forest':     Pipeline([('sc', StandardScaler()), ('m', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))]),
    'XGBoost':           Pipeline([('sc', StandardScaler()), ('m', XGBRegressor(n_estimators=100, random_state=42))]),
}

cv_results = {}
for name, pipe in models_cv.items():
    cv = cross_validate(pipe, X_tr, y_tr, groups=groups, cv=gkf,
                        scoring=['neg_root_mean_squared_error', 'r2'],
                        return_train_score=True)
    cv_rmse = -cv['test_neg_root_mean_squared_error']
    cv_r2   =  cv['test_r2']
    cv_results[name] = cv
    print(f'{name:<26}  CV RMSE: {cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}  '
          f'|  CV R²: {cv_r2.mean():.3f} ± {cv_r2.std():.3f}')

## 3. RandomizedSearchCV — Tune Random Forest

In [ ]:
param_grid_rf = {
    'm__n_estimators':      [50, 100, 200, 300],
    'm__max_depth':         [None, 5, 10, 15],
    'm__min_samples_split': [2, 5, 10],
    'm__min_samples_leaf':  [1, 2, 4],
    'm__max_features':      ['sqrt', 'log2', 0.5],
}

print('Searching Random Forest hyperparameters (20 iterations)...')
rf_pipe = Pipeline([('sc', StandardScaler()),
                    ('m',  RandomForestRegressor(random_state=42, n_jobs=-1))])
rf_search = RandomizedSearchCV(
    rf_pipe, param_grid_rf, n_iter=20,
    cv=GroupKFold(n_splits=5),
    scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1, refit=True,
)
rf_search.fit(X_tr, y_tr, groups=groups)

best_rf  = rf_search.best_estimator_
best_rmse_rf = -rf_search.best_score_
print(f'Best CV RMSE: {best_rmse_rf:.4f}%')
print(f'Best params:')
for k, v in rf_search.best_params_.items():
    print(f'  {k}: {v}')

## 4. RandomizedSearchCV — Tune XGBoost

In [ ]:
param_grid_xgb = {
    'm__n_estimators':     [50, 100, 200, 300],
    'm__max_depth':        [3, 5, 7, 9],
    'm__learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'm__subsample':        [0.7, 0.8, 0.9, 1.0],
    'm__colsample_bytree': [0.6, 0.8, 1.0],
    'm__min_child_weight': [1, 3, 5],
}

print('Searching XGBoost hyperparameters (20 iterations)...')
xgb_pipe = Pipeline([('sc', StandardScaler()),
                     ('m',  XGBRegressor(random_state=42))])
xgb_search = RandomizedSearchCV(
    xgb_pipe, param_grid_xgb, n_iter=20,
    cv=GroupKFold(n_splits=5),
    scoring='neg_root_mean_squared_error',
    random_state=42, n_jobs=-1, refit=True,
)
xgb_search.fit(X_tr, y_tr, groups=groups)

best_xgb      = xgb_search.best_estimator_
best_rmse_xgb = -xgb_search.best_score_
print(f'Best CV RMSE: {best_rmse_xgb:.4f}%')
print(f'Best params:')
for k, v in xgb_search.best_params_.items():
    print(f'  {k}: {v}')

## 5. Final Test-Set Comparison

In [ ]:
all_models = {
    'Linear Regression':    models_cv['Linear Regression'],
    'Random Forest':         models_cv['Random Forest'],
    'XGBoost':               models_cv['XGBoost'],
    'Random Forest (tuned)': best_rf,
    'XGBoost (tuned)':       best_xgb,
}

# Train base models (already cross-validated, now refit on full training data)
for name in ['Linear Regression', 'Random Forest', 'XGBoost']:
    all_models[name].fit(X_tr, y_tr)

results = []
for name, pipe in all_models.items():
    tr_m = compute_metrics(y_tr, pipe.predict(X_tr), name)
    te_m = compute_metrics(y_te, pipe.predict(X_te), name)
    results.append({
        'model': name,
        'MAE_train': tr_m['MAE'],  'RMSE_train': tr_m['RMSE'],  'R2_train': tr_m['R2'],
        'MAE_test':  te_m['MAE'],  'RMSE_test':  te_m['RMSE'],  'R2_test':  te_m['R2'],
    })

table = build_comparison_table(results)
print_metrics_table(table, 'SOH — Full Model Comparison (Baseline + Tuned)')
table

## 6. Save Best SOH Model

In [ ]:
best_row = table.iloc[0]
best_name = best_row['model']
best_pipe = all_models[best_name]

print(f'Best SOH model: {best_name}')
print(f'  Test RMSE: {best_row["RMSE_test"]:.4f}%')
print(f'  Test R²  : {best_row["R2_test"]:.4f}')

path = os.path.join(ROOT, 'models', 'soh', 'best_soh_model.pkl')
joblib.dump(best_pipe, path)
print(f'\nSaved: {path}')

# Save metrics
os.makedirs(os.path.join(ROOT, 'results', 'metrics'), exist_ok=True)
table.to_csv(os.path.join(ROOT, 'results', 'metrics', 'soh_metrics.csv'))
print('Saved: results/metrics/soh_metrics.csv')

## 7. Prediction on Test Set — Best Model

In [ ]:
y_pred = best_pipe.predict(X_te)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_te, y_pred, alpha=0.4, s=20, color='steelblue', edgecolors='k', linewidths=0.2)
lim = [max(0, y_te.min()-5), y_te.max()+5]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Perfect fit')
te_m = compute_metrics(y_te, y_pred, best_name)
axes[0].set_title(f'SOH Actual vs Predicted\n{best_name}  (RMSE={te_m["RMSE"]:.3f}%  R²={te_m["R2"]:.3f})')
axes[0].set_xlabel('Actual SOH (%)')
axes[0].set_ylabel('Predicted SOH (%)')
axes[0].legend()

residuals = y_te - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.4, s=20, color='#C62828', edgecolors='k', linewidths=0.2)
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted SOH (%)')
axes[1].set_ylabel('Residual (Actual − Predicted)')
axes[1].set_title('SOH Residual Plot')

plt.tight_layout()
plt.show()